# 00 - ViT dynamic-LRP repro (toolchain-proof gate)

Reproduces the reference `keeinlev/dynamicLRP` `src/experiments/ViT.ipynb`
attribution flow on the **pinned** MapClass stack (torch==2.7.1,
transformers==4.52.3, dynamicLRP vendored at SHA
`405e74243ecaa1f615f418fdc8ba24c3c5889b1e`).

This is ROADMAP Success Criterion 1 / Plan 01-01 Task 2. The SigLIP-2 swap
(Plan 03) must NOT begin until this notebook reproduces a structured ViT
relevance heatmap on this exact environment.

Flow mirrors the reference notebook cells 1-15 verbatim in mechanism
(`ViTForImageClassification` 224/patch16, one CIFAR10 image, `LRPEngine(
use_gamma=True, no_recompile=True)`, `params_to_interpret=[img_tensor]`,
`run(output.logits)`).

In [1]:
# --- Wire the vendored dynamicLRP engine onto sys.path (D-08: in-tree, offline) ---
import os
import sys
from pathlib import Path

# notebooks/ -> repo root
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
VENDORED_LRP_SRC = REPO_ROOT / "third_party" / "dynamicLRP" / "src"
PROJECT_SRC = REPO_ROOT / "src"
for p in (str(VENDORED_LRP_SRC), str(PROJECT_SRC)):
    if p not in sys.path:
        sys.path.insert(0, p)

vendor_sha = (REPO_ROOT / "third_party" / "dynamicLRP" / "VENDOR_SHA").read_text().strip()
print("repo root        :", REPO_ROOT)
print("vendored LRP src :", VENDORED_LRP_SRC)
print("VENDOR_SHA       :", vendor_sha)
assert vendor_sha == "405e74243ecaa1f615f418fdc8ba24c3c5889b1e", vendor_sha

repo root        : /home/drdreadknee/mapclass
vendored LRP src : /home/drdreadknee/mapclass/third_party/dynamicLRP/src
VENDOR_SHA       : 405e74243ecaa1f615f418fdc8ba24c3c5889b1e


In [2]:
import random

import numpy as np
import torch
import torchvision.datasets as datasets
import torchvision.transforms as T
import transformers
from matplotlib import pyplot as plt
from transformers import ViTConfig, ViTForImageClassification

# Pinned-stack assertion (fail loudly if the env drifted)
assert torch.__version__.startswith("2.7.1"), torch.__version__
assert transformers.__version__ == "4.52.3", transformers.__version__
print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)
print("cuda        :", torch.cuda.is_available())

seed_value = 42
torch.manual_seed(seed_value)
np.random.seed(seed_value)
random.seed(seed_value)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

/home/drdreadknee/mapclass/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch       : 2.7.1+cu126
transformers: 4.52.3
cuda        : False


In [3]:
# Vendored engine import (offline; no network, no clone-at-setup)
from lrp_engine import LRPEngine

print("LRPEngine imported from:", LRPEngine.__module__)
assert hasattr(LRPEngine, "run") and hasattr(LRPEngine, "get_model_operations")

LRPEngine imported from: lrp_engine.lrp


In [4]:
# Reference model: HF ViTForImageClassification, patch16 / 224 (ViT.ipynb cell 3)
vit_model = ViTForImageClassification.from_pretrained(
    "nateraw/vit-base-patch16-224-cifar10"
)
vit_model.to(device)
vit_model.eval()
print("model loaded:", type(vit_model).__name__)

model loaded: ViTForImageClassification


In [5]:
# One sample image, reference transform (ViT.ipynb cell 6)
transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
dataset = datasets.CIFAR10(
    root=str(REPO_ROOT / ".cache" / "cifar10"),
    train=False,
    download=True,
    transform=transform,
)
sample_img, sample_label = dataset[0]
print("sample image tensor:", tuple(sample_img.shape), "label:", sample_label)

100%|██████████| 170M/170M [00:04<00:00, 36.7MB/s] 


sample image tensor: (3, 224, 224) label: 3


In [6]:
# Forward + coverage probe (ViT.ipynb cells 11-12)
# img_tensor is the SAME object passed to both the forward and params_to_interpret.
img_tensor = sample_img.unsqueeze(0).to(device).requires_grad_()
output = vit_model(img_tensor)

op_names, op_count, _graph = LRPEngine.get_model_operations(output.logits)
print("get_model_operations op count:", op_count)
assert op_count > 0, "coverage probe returned no autograd ops"

get_model_operations op count: 16


In [7]:
# Dynamic LRP (ViT.ipynb cells 10, 13) - target is the 2-D classification logits
engine = LRPEngine(use_gamma=True, no_recompile=True)
engine.params_to_interpret = [img_tensor]
ckpt_vals, param_vals = engine.run(output.logits)

relevance = param_vals[0]            # positionally matches params_to_interpret[0]
print("relevance type :", type(relevance))
print("relevance shape:", tuple(relevance.shape))
assert tuple(relevance.shape) == tuple(img_tensor.shape), (
    relevance.shape, img_tensor.shape
)

relevance type : <class 'torch.Tensor'>
relevance shape: (1, 3, 224, 224)


In [ ]:
# Per-patch grid reshape - verified ViT geometry: 224 / 16 = 14 -> 14x14 patches
patch_size = 16
img_dims = 224
grid = img_dims // patch_size            # = 14
assert img_dims % patch_size == 0

# LRP relevance is SIGNED: positive = evidence FOR the predicted class at that
# location, negative = evidence AGAINST. We keep the sign (consistent with the
# raw_heatmap in the next cell). The earlier .abs() reduction collapsed the
# sign and made the diverging colormap unreadable (bounded at zero).
rel = relevance.detach()[0]                       # (3,224,224)

# Signed pixel relevance: sum over channels (matches ViT.ipynb cell 15).
pixel_rel_signed = rel.sum(0)                      # (224,224), signed
# Magnitude pixel relevance: abs then sum (where attribution concentrates).
pixel_rel_mag = rel.abs().sum(0)                   # (224,224), >= 0

# block-reduce 224x224 -> 14x14 (per-patch mean), preserving each reduction.
patch_grid = (
    pixel_rel_signed.reshape(grid, patch_size, grid, patch_size).mean((1, 3))
)
patch_grid_mag = (
    pixel_rel_mag.reshape(grid, patch_size, grid, patch_size).mean((1, 3))
)
print("patch grid shape:", tuple(patch_grid.shape))
assert tuple(patch_grid.shape) == (14, 14), patch_grid.shape


In [ ]:
# Visualize (ViT.ipynb cell 15) - SIGNED relevance on a zero-centered diverging
# colormap so negative relevance (evidence AGAINST the class) is visible, plus a
# magnitude panel showing where attribution concentrates regardless of sign.
import numpy as np
from matplotlib.colors import TwoSlopeNorm

raw_heatmap = pixel_rel_signed.cpu().numpy()       # signed (224,224)

# de-normalize the sample image just for display
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
disp_img = (sample_img * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()


def _sym_norm(arr):
    """Zero-centered symmetric norm so white==0 on the bwr diverging map."""
    m = float(np.nanmax(np.abs(arr)))
    if not np.isfinite(m) or m == 0.0:
        m = 1e-12
    return TwoSlopeNorm(vmin=-m, vcenter=0.0, vmax=m)


pg = patch_grid.cpu().numpy()
pg_mag = patch_grid_mag.cpu().numpy()

fig, axs = plt.subplots(1, 4, figsize=(20, 5))

axs[0].imshow(disp_img)
axs[0].set_title(f"CIFAR10 sample (label {sample_label})")
axs[0].set_axis_off()

axs[1].imshow(disp_img)
im1 = axs[1].imshow(raw_heatmap, cmap="bwr", norm=_sym_norm(raw_heatmap), alpha=0.5)
axs[1].set_title("Signed relevance (overlay)")
axs[1].set_axis_off()
fig.colorbar(im1, ax=axs[1], fraction=0.046)

im2 = axs[2].imshow(pg, cmap="bwr", norm=_sym_norm(pg), interpolation="nearest")
axs[2].set_title("14x14 signed per-patch (bwr, 0-centered)")
axs[2].set_axis_off()
fig.colorbar(im2, ax=axs[2], fraction=0.046)

im3 = axs[3].imshow(pg_mag, cmap="magma", interpolation="nearest")
axs[3].set_title("14x14 |relevance| magnitude")
axs[3].set_axis_off()
fig.colorbar(im3, ax=axs[3], fraction=0.046)

fig.suptitle(
    f"ViT dynamic-LRP repro | torch {torch.__version__} | "
    f"transformers {transformers.__version__} | LRP @ {vendor_sha[:12]}"
)
fig.tight_layout()
plt.show()

# Structure probe (NOT a gate - the human eyeball is the gate). Signed range
# MUST straddle zero for a meaningful LRP map; magnitude is >= 0 by definition.
print("signed patch min/max :", float(patch_grid.min()), float(patch_grid.max()))
print("has negative relevance:", bool((patch_grid < 0).any().item()))
print("|relevance| patch max :", float(patch_grid_mag.max()))
print("relevance std         :", float(relevance.detach().std()))


## Human-verify gate (Plan 01-01 Task 3)

Restart Kernel & Run All against the `mapclass (.venv)` kernel and confirm the
overlay/patch-grid heatmap is **structured and non-uniform** (concentrates on
the object, not flat / not pure noise) - matching the qualitative look of the
reference dynamicLRP `ViT.ipynb` output. If so, the pinned toolchain is proven
and Plan 03 (SigLIP-2 swap) may proceed.